# Week 3 - Spark Transform
Đọc dataset 1.2GB bằng Spark, so sánh với pandas, sau đó transform bằng DataFrame API và Spark SQL ( lặp lại các câu hỏi phân tích đã làm ở Week 1-2 nhưng bằng Spark.)

In [1]:
import os
import time
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

DATA_PATH = os.path.expanduser("~/intern_DE/Week_3/data/raw/ecommerce_sales_large.csv")

spark = SparkSession.builder \
    .appName("Week3-Transform") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark

26/07/25 22:02:56 WARN Utils: Your hostname, Hvh214 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/25 22:02:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/25 22:02:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1. So sánh thời gian đọc: Spark vs pandas
Chạy pandas trước để có baseline ( pandas load toàn bộ vào RAM, nếu máy ít RAM có thể chậm hoặc lỗi MemoryError với file 1.2GB.)

In [2]:
import pandas as pd

start = time.time()
df_pandas = pd.read_csv(DATA_PATH)
pandas_time = time.time() - start
print(f"[pandas] Đọc {len(df_pandas):,} dòng trong {pandas_time:.2f} giây")

[pandas] Đọc 17,500,000 dòng trong 14.72 giây


In [3]:
start = time.time()
df_spark = spark.read.csv(DATA_PATH, header=True, inferSchema=True)
row_count = df_spark.count()  # Spark lazy evaluation -- cần action (count) để trigger job thật sự
spark_time = time.time() - start
print(f"[spark] Đọc {row_count:,} dòng trong {spark_time:.2f} giây")

[spark] Đọc 17,500,000 dòng trong 16.81 giây


**Lưu ý quan trọng về kết quả so sánh này:**

Spark có overhead khởi tạo JVM + lên kế hoạch thực thi (lazy evaluation), nên với 1 lần đọc đơn giản như trên, Spark **chưa chắc đã nhanh hơn pandas** *đặc biệt khi chạy local trên 1 máy. Lợi thế thật sự của Spark chỉ rõ ràng khi:
- Dataset lớn hơn RAM máy (pandas sẽ crash, Spark vẫn chạy được nhờ xử lý theo partition)
- Có nhiều phép biến đổi phức tạp nối tiếp nhau (Spark tối ưu execution plan tổng thể thay vì chạy tuần tự từng bước như pandas)

In [4]:
df_spark.printSchema()
df_spark.show(5)

root
 |-- order_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_category: string (nullable = true)
 |-- region: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- delivery_days: integer (nullable = true)
 |-- customer_rating: double (nullable = true)
 |-- revenue: double (nullable = true)

+--------+----------+-----------+----------------+------+--------+----------+--------+--------------+-------------+---------------+-------+
|order_id|order_date|customer_id|product_category|region|quantity|unit_price|discount|payment_method|delivery_days|customer_rating|revenue|
+--------+----------+-----------+----------------+------+--------+----------+--------+--------------+-------------+---------------+-------+
|       1|2030-06-26|      29918|          Beauty| South|     

## 2. Transform bằng DataFrame API

In [ ]:
# Doanh thu theo category
revenue_by_category = (
    df_spark.groupBy("product_category")
    .agg(F.sum("revenue").alias("total_revenue"), F.count("*").alias("order_count"))
    .orderBy(F.desc("total_revenue"))
)
revenue_by_category.show()

+----------------+---------------+-----------+
|product_category|  total_revenue|order_count|
+----------------+---------------+-----------+
|        Clothing|9.40603658058E9|    7003535|
|     Electronics|7.04796802814E9|    5250023|
|          Beauty|7.03634070846E9|    5246442|
+----------------+---------------+-----------+



In [6]:
# Doanh thu theo region, theo năm
df_spark_with_year = df_spark.withColumn("order_year", F.year("order_date"))

revenue_by_region_year = (
    df_spark_with_year.groupBy("region", "order_year")
    .agg(F.sum("revenue").alias("total_revenue"))
    .orderBy("order_year", "region")
)
revenue_by_region_year.show(10)

+------+----------+--------------------+
|region|order_year|       total_revenue|
+------+----------+--------------------+
|  East|      2022| 3.863822487099998E8|
| North|      2022| 4.698133874800005E8|
| South|      2022| 4.528368490900004E8|
|  West|      2022| 3.683524615799999E8|
|  East|      2023|3.8575942509999985E8|
| North|      2023| 4.704257498400011E8|
| South|      2023| 4.518290733599997E8|
|  West|      2023| 3.682869604200001E8|
|  East|      2024|  3.87695477570001E8|
| North|      2024| 4.705588090900016E8|
+------+----------+--------------------+
only showing top 10 rows



## 3. Transform bằng Spark SQL
So sánh cú pháp với câu query tương đương đã viết ở `Week_1/sql/analysis.sql`.

In [11]:
df_spark_with_year.createOrReplaceTempView("sales")

result = spark.sql("""
    SELECT product_category, SUM(revenue) AS total_revenue, COUNT(*) AS order_count
    FROM sales
    GROUP BY product_category
    ORDER BY total_revenue DESC
""")
result.show()

+----------------+-------------------+-----------+
|product_category|      total_revenue|order_count|
+----------------+-------------------+-----------+
|        Clothing|9.406036580579933E9|    7003535|
|     Electronics|7.047968028139965E9|    5250023|
|          Beauty| 7.03634070845997E9|    5246442|
+----------------+-------------------+-----------+



In [12]:
top_customers = spark.sql("""
    SELECT customer_id, SUM(revenue) AS total_spent, COUNT(*) AS order_count
    FROM sales
    GROUP BY customer_id
    ORDER BY total_spent DESC
    LIMIT 10
""")
top_customers.show()

+-----------+-----------------+-----------+
|customer_id|      total_spent|order_count|
+-----------+-----------------+-----------+
|      11497|632498.6200000001|        417|
|      18284|627065.7000000001|        424|
|       4321|622837.8700000001|        432|
|      11848|         616799.2|        428|
|      29136|612312.4400000001|        412|
|      37588|        609438.35|        409|
|       6769|        608957.36|        399|
|       8767|607148.9299999999|        433|
|      17360|        606343.64|        386|
|       6170|605837.8199999998|        397|
+-----------+-----------------+-----------+



**So sánh cú pháp với Week 1 (SQL thuần trên MySQL):**

```sql
SELECT product_category, SUM(revenue) AS total_revenue
FROM orders
GROUP BY product_category;
```

→ Spark SQL gần như giống hệt cú pháp SQL chuẩn, khác biệt chính không nằm ở câu lệnh mà ở **cách thực thi phía sau**: MySQL chạy trên 1 engine đơn, còn Spark SQL biên dịch câu query thành execution plan chạy phân tán trên nhiều partition/executor (dù ở đây là local mode nên chỉ có 1 máy, nhưng vẫn chia nhiều task chạy song song trên các CPU core).

## 4. Xem execution plan của Spark (tương đương EXPLAIN trong SQL)

In [13]:
result.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (7)
+- Sort (6)
   +- Exchange (5)
      +- HashAggregate (4)
         +- Exchange (3)
            +- HashAggregate (2)
               +- Scan csv  (1)


(1) Scan csv 
Output [2]: [product_category#20, revenue#28]
Batched: false
Location: InMemoryFileIndex [file:/home/inuc214/intern_DE/Week_3/data/raw/ecommerce_sales_large.csv]
ReadSchema: struct<product_category:string,revenue:double>

(2) HashAggregate
Input [2]: [product_category#20, revenue#28]
Keys [1]: [product_category#20]
Functions [2]: [partial_sum(revenue#28), partial_count(1)]
Aggregate Attributes [2]: [sum#369, count#370L]
Results [3]: [product_category#20, sum#371, count#372L]

(3) Exchange
Input [3]: [product_category#20, sum#371, count#372L]
Arguments: hashpartitioning(product_category#20, 200), ENSURE_REQUIREMENTS, [plan_id=446]

(4) HashAggregate
Input [3]: [product_category#20, sum#371, count#372L]
Keys [1]: [product_category#20]
Functions [2]: [sum(revenue#28), count(1)]
Aggregat

## 5. Lưu kết quả transform (dùng cho bước convert Parquet/ORC ở notebook benchmark_formats)

In [14]:
OUTPUT_DIR = os.path.expanduser("~/intern_DE/Week_3/data/processed")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Lưu dataset đã thêm cột order_year, dùng cho bước convert format + partitioning tiếp theo
df_spark_with_year.write.mode("overwrite").csv(
    os.path.join(OUTPUT_DIR, "sales_with_year_csv"), header=True
)
print("Đã lưu xong.")

Đã lưu xong.


## 6. Ghi lại thời gian benchmark vào results/

In [15]:
import csv

benchmark_row = {
    "step": "read_full_dataset",
    "pandas_seconds": round(pandas_time, 2),
    "spark_seconds": round(spark_time, 2),
    "row_count": row_count,
}

results_path = os.path.expanduser("~/intern_DE/Week_3/results/benchmark_results.csv")
file_exists = os.path.exists(results_path)

with open(results_path, "a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=benchmark_row.keys())
    if not file_exists:
        writer.writeheader()
    writer.writerow(benchmark_row)

print("Đã ghi vào", results_path)

Đã ghi vào /home/inuc214/intern_DE/Week_3/results/benchmark_results.csv


In [16]:
spark.stop()